# RAG Module · Notebook 1 · Why RAG?
Understand the two problems RAG was invented to solve: hallucination and context limits.

---
> **Setup:** Run the first two cells before anything else.
> API key is loaded automatically from the `.env` file in the parent folder.

In [1]:
%pip install -q google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, json, math, time, base64, getpass, re, urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # picks up modules/.env symlink
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

client = genai.Client(api_key=api_key)
MODEL = 'gemma-4-31b-it'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen    = client.models.generate_content
_raw_stream = client.models.generate_content_stream
_raw_embed  = client.models.embed_content
_raw_count  = client.models.count_tokens
client.models.generate_content        = lambda *a,**kw: _call_with_retry(_raw_gen,    *a,**kw)
client.models.generate_content_stream = lambda *a,**kw: _call_with_retry(_raw_stream, *a,**kw)
client.models.embed_content           = lambda *a,**kw: _call_with_retry(_raw_embed,  *a,**kw)
client.models.count_tokens            = lambda *a,**kw: _call_with_retry(_raw_count,  *a,**kw)

print('✅ Setup complete. Model:', MODEL, '| Retry-on-429: enabled')

✅ Setup complete. Model: gemma-4-31b-it | Retry-on-429: enabled


### 🔌 API Key Test

In [3]:
try:
    _r = client.models.generate_content(
        model=MODEL,
        contents="Reply with exactly the words: Hello LLM course",
        config=cfg(temperature=0.0)
    )
    print("✅ API key working!")
    print("Model says:", get_text(_r))
except Exception as e:
    print("❌ API key error:", e)

✅ API key working!
Model says: Hello LLM course


---
## 1. Hallucination — When the Model Confidently Makes Things Up
LLMs generate plausible text, not factually verified text — a critical distinction.

**What is hallucination?** Hallucination is when the model produces confident, fluent, but factually wrong output. It is a structural consequence of next-token prediction: the model predicts what *sounds* right, not what *is* true. There is no internal fact-checker — only a probability distribution over the next word.

**Developer analogy:** A student who didn't study but writes a convincing essay — fluent, well-structured, and completely wrong.

```
Prompt: "What was AcmeCorp's Q3 2024 revenue?"
         ↓
LLM: [no AcmeCorp data in training] → predicts plausible-sounding number
         ↓
Output: "$47.3 million"   ← invented. confident. wrong.
```

### Common Hallucination Patterns

| Hallucination Pattern | Example |
|---|---|
| Invented statistics | Made-up revenue figures, percentages |
| Fabricated citations | Fake paper titles, non-existent authors |
| Wrong technical details | Incorrect API signatures, wrong dates |
| Plausible but false events | Events that didn't happen, described vividly |

In [4]:
# Ask about a fictional private company — model has no real data
r = client.models.generate_content(
    model=MODEL,
    contents="What was AcmeCorp's total revenue in Q3 2024 and who is their current CEO?",
    config=cfg(temperature=0.7)
)
print("Without any context:")
print("-" * 40)
print(get_text(r))
print("\n⚠️  Notice: the model sounds confident — but these figures are invented.")

Without any context:
----------------------------------------
**AcmeCorp** is a fictional company (often used as a placeholder name in examples, cartoons, and legal documents). Because it is not a real entity, it does not have actual financial reports, revenue for Q3 2024, or a real-world CEO.

If you are referring to a specific company from a book, movie, game, or a real company with a similar name, please provide more details and I would be happy to help you find the information!

⚠️  Notice: the model sounds confident — but these figures are invented.


### The Fix: Grounding

The fix is to give the model the facts directly in the prompt and instruct it to use **only** those facts. This is called **grounding** — you anchor the model's output to a specific, trusted source of truth.

The grounding pattern:
1. Collect the relevant facts (from a database, document, or API)
2. Include them verbatim in the prompt
3. Instruct the model to answer only from those facts, and refuse if the answer isn't there

RAG automates step 1 at scale — but the grounding pattern stays the same.

In [5]:
# Grounding: provide the actual facts in the prompt
company_context = """
AcmeCorp Financial Summary:
- Q3 2024 Revenue: $12.8 million (down 3% YoY)
- Q3 2024 Net Loss: $1.2 million
- CEO: Sarah Chen (appointed January 2023)
- Headcount: 87 employees
"""

r = client.models.generate_content(
    model=MODEL,
    contents=f"""Use ONLY the information in the document below to answer the question.
If the answer is not in the document, say exactly: "This information is not in the provided document."

Document:
{company_context}

Question: What was AcmeCorp's Q3 2024 revenue and who is their CEO?""",
    config=cfg(temperature=0.0)
)
print("With grounded context:")
print("-" * 40)
print(get_text(r))

With grounded context:
----------------------------------------
AcmeCorp's Q3 2024 revenue was $12.8 million and their CEO is Sarah Chen.


In [6]:
# Test the grounding boundary — ask something NOT in the document
r = client.models.generate_content(
    model=MODEL,
    contents=f"""Use ONLY the information in the document below to answer the question.
If the answer is not in the document, say exactly: "This information is not in the provided document."

Document:
{company_context}

Question: What are AcmeCorp's main products?""",
    config=cfg(temperature=0.0)
)
print("Out-of-scope question:")
print("-" * 40)
print(get_text(r))
print("\n✅ Model correctly refuses to hallucinate beyond the provided context.")

Out-of-scope question:
----------------------------------------
This information is not in the provided document.

✅ Model correctly refuses to hallucinate beyond the provided context.


### Hallucination Mitigation Strategies

| Strategy | When to use |
|---|---|
| Lower temperature | Reduce creativity/randomness for factual tasks |
| Grounding (context injection) | You have the relevant facts to include in the prompt |
| RAG | Facts are in a large document corpus — retrieve relevant chunks automatically |
| Verification step | Ask the model to fact-check its own answer against provided sources |

---
## 2. Context Window Limits — The Model Can't Read Everything at Once
Every model has a hard limit on how many tokens it can process in a single call.

**What is the context window?** The context window is the total number of tokens the model can see at once — input *and* output combined. Tokens that fall outside this window are simply invisible to the model. It cannot reason about content it cannot see.

**Developer analogy:** Reading through a porthole — you can only see what fits in the opening, no matter how long the document is.

### Context Window Sizes

| Model | Context Window | Human-scale equivalent |
|---|---|---|
| Gemma 4 31B | ~128k tokens | ~100,000 words / a full novel |
| Gemini 2.5 Flash | 1M tokens | ~750,000 words / 10+ novels |
| GPT-4o | 128k tokens | ~100,000 words |

Even a 1M-token context window cannot hold an enterprise knowledge base, a full codebase, or years of support tickets. And even when the document *fits*, packing irrelevant content into the context degrades answer quality.

In [7]:
# Show how token counts scale with document size
import math

base_sentence = "The quick brown fox jumps over the lazy dog. "

# Approximate different document sizes
sizes = [100, 1_000, 10_000, 50_000]
gemma_limit = 128_000

print(f"{'Words':>10} | {'~Tokens':>10} | {'% of Gemma 4 128k':>20}")
print("-" * 48)
for target_words in sizes:
    text = base_sentence * math.ceil(target_words / len(base_sentence.split()))
    text = ' '.join(text.split()[:target_words])
    result = client.models.count_tokens(model=MODEL, contents=text)
    tokens = result.total_tokens
    pct = tokens / gemma_limit * 100
    print(f"{target_words:>10,} | {tokens:>10,} | {pct:>19.1f}%")

     Words |    ~Tokens |    % of Gemma 4 128k
------------------------------------------------
       100 |        112 |                 0.1%


     1,000 |      1,112 |                 0.9%


    10,000 |     11,112 |                 8.7%


    50,000 |     55,556 |                43.4%


### The Naive Approach: Prompt Stuffing

The obvious solution is to paste the entire document into the prompt. This works for short documents — but at scale it breaks down in three ways:

1. **Hits context limit** — large corpora exceed even the largest context windows
2. **Slow and expensive** — billing is per-token; sending 50k tokens every query is costly
3. **Signal buried in noise** — the model has to find the relevant sentence in a sea of irrelevant text, which degrades accuracy

The solution is to retrieve only the relevant fragments — the core idea behind RAG.

In [8]:
# Show the cost of stuffing a large document vs. targeted retrieval
long_doc_words = 50_000
long_doc = (base_sentence * math.ceil(long_doc_words / len(base_sentence.split())))
long_doc = ' '.join(long_doc.split()[:long_doc_words])

full_doc_tokens = client.models.count_tokens(model=MODEL, contents=long_doc).total_tokens

# A targeted chunk is ~300 words
chunk = ' '.join(long_doc.split()[:300])
chunk_tokens = client.models.count_tokens(model=MODEL, contents=chunk).total_tokens

print(f"Full document:     {full_doc_tokens:,} tokens ({full_doc_tokens/gemma_limit*100:.1f}% of context)")
print(f"One relevant chunk: {chunk_tokens:,} tokens ({chunk_tokens/gemma_limit*100:.2f}% of context)")
print(f"\nRAG uses ~{full_doc_tokens//chunk_tokens}x fewer tokens per query.")

Full document:     55,556 tokens (43.4% of context)
One relevant chunk: 334 tokens (0.26% of context)

RAG uses ~166x fewer tokens per query.


### Strategies for Large Document Corpora

| Strategy | Scales to large corpus? | Cost | Quality |
|---|---|---|---|
| Full-context stuffing | No (hits limit) | High | Noisy |
| Summarization chain | Partially | Medium | Loses detail |
| RAG (retrieve top-K) | Yes | Low | High (relevant only) |

---
## 3. Chunking — Splitting Documents Into Retrievable Pieces
Before storing documents in a vector database, we split them into chunks the model can actually use.

**Developer analogy:** Cutting a textbook into index cards — each card covers one topic, small enough to find and hand to someone quickly.

### Two failure modes to avoid

- **Chunks too large** → you retrieve irrelevant noise along with the answer; the model gets confused
- **Chunks too small** → you cut mid-sentence or mid-idea; the model gets incomplete context

The goal is chunks that are *semantically coherent* and *small enough to be specific*.

In [9]:
def chunk_fixed(text, chunk_size=200):
    """Split on character count — simple but can cut mid-word."""
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

def chunk_overlap(text, chunk_size=200, overlap=50):
    """Overlapping chunks — preserves context across boundaries."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# Sample document — about Python (self-contained, no external files)
sample_doc = """Python is a high-level, general-purpose programming language created by Guido van Rossum
and first released in 1991. Its design philosophy emphasizes code readability and simplicity, allowing
developers to express concepts in fewer lines of code than languages like C++ or Java. Python supports
multiple programming paradigms including procedural, object-oriented, and functional programming.
The Python Package Index (PyPI) hosts hundreds of thousands of third-party libraries. Popular frameworks
include Django and Flask for web development, NumPy and Pandas for data science, and TensorFlow and
PyTorch for machine learning. Python consistently ranks as one of the most popular programming languages
in developer surveys. It is widely used in web development, data analysis, artificial intelligence,
scientific computing, and automation scripting."""

fixed_chunks = chunk_fixed(sample_doc, chunk_size=200)
overlap_chunks = chunk_overlap(sample_doc, chunk_size=200, overlap=50)

print(f"Fixed chunking (200 chars):     {len(fixed_chunks)} chunks")
print(f"Overlapping (200 chars, 50 ov): {len(overlap_chunks)} chunks")
print()
print("=== Fixed chunk #1 ===")
print(repr(fixed_chunks[0]))
print()
print("=== Fixed chunk #2 ===")
print(repr(fixed_chunks[1]))
print()
print("=== Overlap chunk #1 ===")
print(repr(overlap_chunks[0]))
print()
print("=== Overlap chunk #2 (notice shared text at boundary) ===")
print(repr(overlap_chunks[1]))

Fixed chunking (200 chars):     5 chunks
Overlapping (200 chars, 50 ov): 6 chunks

=== Fixed chunk #1 ===
'Python is a high-level, general-purpose programming language created by Guido van Rossum\nand first released in 1991. Its design philosophy emphasizes code readability and simplicity, allowing\ndevelope'

=== Fixed chunk #2 ===
'rs to express concepts in fewer lines of code than languages like C++ or Java. Python supports\nmultiple programming paradigms including procedural, object-oriented, and functional programming.\nThe Pyt'

=== Overlap chunk #1 ===
'Python is a high-level, general-purpose programming language created by Guido van Rossum\nand first released in 1991. Its design philosophy emphasizes code readability and simplicity, allowing\ndevelope'

=== Overlap chunk #2 (notice shared text at boundary) ===
'code readability and simplicity, allowing\ndevelopers to express concepts in fewer lines of code than languages like C++ or Java. Python supports\nmultiple programming p

In [10]:
# ✏️ Try your own chunking settings
my_text = "Replace this with your own text to experiment with chunking."
my_chunk_size = 150
my_overlap = 30

my_chunks = chunk_overlap(my_text, chunk_size=my_chunk_size, overlap=my_overlap)
print(f"Your text: {len(my_text)} chars → {len(my_chunks)} chunks")
for i, c in enumerate(my_chunks):
    print(f"\nChunk {i+1}: {repr(c)}")

Your text: 60 chars → 1 chunks

Chunk 1: 'Replace this with your own text to experiment with chunking.'


### Recommended Chunking Settings

| Use Case | Chunk Size | Overlap | Reasoning |
|---|---|---|---|
| Q&A over documents | 200–400 chars | 40–80 chars | Small enough to be specific |
| Long-form summarization | 800–1200 chars | 100–200 chars | Preserve paragraph context |
| Code search | 300–600 chars | 50 chars | One function/block per chunk |
| News / short articles | 150–300 chars | 30–50 chars | Articles are already dense |

---
## 4. The Case for RAG — Solving Both Problems at Once
RAG combines grounding and efficient retrieval to make LLMs reliable on any document corpus.

**The two problems, revisited:**
- **Hallucination** — the model invents facts when it doesn't have relevant knowledge
- **Context limits** — you can't fit an entire document corpus into a single prompt

RAG solves both simultaneously: instead of stuffing everything in or relying on memory, it *retrieves* only the relevant pieces and *injects* them as grounded context.

**Developer analogy:** An open-book exam where the retrieval system automatically finds the right page — the model answers from evidence, not memory.

### The Full RAG Pipeline

```
                    ┌─────────────────────────────────┐
  INDEXING          │  Documents → Chunk → Embed       │
  (done once)       │            → Store in Vector DB  │
                    └─────────────────────────────────┘

                    ┌─────────────────────────────────┐
  QUERYING          │  Question → Embed                │
  (every request)   │          → Find top-K chunks     │
                    │          → Build augmented prompt │
                    │          → LLM generates answer  │
                    └─────────────────────────────────┘
```

> **Note:** You'll build exactly this pipeline in Notebook 3.

### RAG vs. Alternatives

| Approach | Knowledge cutoff | Cost | Scales to large corpus | Best for |
|---|---|---|---|---|
| Bare LLM | Training cutoff | Low | N/A | General knowledge tasks |
| Full-context stuffing | Up to date | High | No | Short documents only |
| Fine-tuning | Updated at fine-tune time | Very high | Partially | Changing model behaviour |
| **RAG** | **Always up to date** | **Low** | **Yes** | **Private / large / changing knowledge** |

---
### Key Takeaways

| Concept | One-liner |
|---|---|
| Hallucination | Model predicts plausible text — not verified facts |
| Grounding | Inject the facts; instruct model to use only them |
| Context window | Hard token limit — can't fit an entire corpus in one call |
| Chunking | Split documents into retrievable pieces before indexing |
| RAG | Retrieve relevant chunks, inject as context, generate grounded answer |

**Next:** Notebook 2 covers embeddings and semantic search — the mechanism that makes retrieval work.